# Resume RAG Experiments

This notebook documents the Milestone 2 resume ingestion and job matching pipeline. It uses the actual project modules: `resume_parser.py`, `resume_rag.py`, `job_matcher.py`, and `evaluation.py`.

In [1]:
import os, sys, json, time
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import fs_tools
import resume_rag
from resume_parser import parse_resume
from job_matcher import match_job_description, extract_job_requirements
import evaluation

## 1. Dataset Overview

The dataset contains fictional resumes in `resumes/` across backend, frontend, full stack, data, ML, DevOps, cloud, security, QA, mobile, product, and business roles. Generated artifacts such as `*_summary.txt` are excluded during ingestion.

In [2]:
discovery = resume_rag.discover_resume_files()
resume_files = discovery['files']
len(resume_files), [f['name'] for f in resume_files[:10]]

(36,
 ['resume_aaron_white.txt',
  'resume_aisha_khan.txt',
  'resume_ben_carter.txt',
  'resume_chloe_nguyen.txt',
  'resume_daniel_evans.txt',
  'resume_david_okafor.txt',
  'resume_elena_popov.txt',
  'resume_emma_wilson.txt',
  'resume_ethan_kim.txt',
  'resume_fatima_hassan.txt'])

## 2. Resume Parsing Example

`resume_parser.py` extracts deterministic metadata: candidate name, skills, estimated years of experience, education, and logical sections. No LLM is used for parsing.

In [3]:
sample_path = os.path.join(PROJECT_DIR, 'resumes', 'resume_emma_wilson.txt')
sample_text = fs_tools.read_file(sample_path)['content']
parsed = parse_resume(sample_text)
parsed

{'name': 'Emma Wilson',
 'skills': ['Python',
  'SQL',
  'JavaScript',
  'FastAPI',
  'Django',
  'Flask',
  'AWS',
  'Docker',
  'Kubernetes',
  'PostgreSQL',
  'Redis'],
 'experience_years': 7,
 'education': 'B.S. Computer Science, Northeastern University, 2017',
 'sections': {'header': 'Emma Wilson\nSenior Python Backend Engineer\nEmail: emma.wilson@example.com | Phone: (555) 101-2001\nLocation: Boston, MA',
  'summary': 'Backend engineer with 7 years of experience building APIs and event-driven services.\nStrong in Python, FastAPI, PostgreSQL, Docker, and AWS.',
  'skills': '- Languages: Python, SQL, JavaScript\n- Frameworks: FastAPI, Django, Flask\n- Cloud: AWS, Docker, Kubernetes\n- Databases: PostgreSQL, Redis',
  'experience': 'Senior Backend Engineer, Northstar Health (2021 - Present)\n- Built FastAPI services for patient data workflows on AWS ECS.\n- Tuned PostgreSQL queries and reduced API latency by 35%.\n\nBackend Engineer, LedgerWorks (2017 - 2021)\n- Developed Python ser

## 3. Section-Aware Chunking Example

The pipeline preserves resume sections first. Small sections remain intact; large sections are split with LangChain's recursive text splitter. This keeps chunks tied to resume structure rather than arbitrary whole-document slices.

In [4]:
chunks = resume_rag.create_resume_chunks(parsed, sample_path)
[(chunk['metadata']['section'], chunk['text'][:120]) for chunk in chunks]

[('header',
  'Emma Wilson\nSenior Python Backend Engineer\nEmail: emma.wilson@example.com | Phone: (555) 101-2001\nLocation: Boston, MA'),
 ('summary',
  'Backend engineer with 7 years of experience building APIs and event-driven services.\nStrong in Python, FastAPI, PostgreS'),
 ('skills',
  '- Languages: Python, SQL, JavaScript\n- Frameworks: FastAPI, Django, Flask\n- Cloud: AWS, Docker, Kubernetes\n- Databases: '),
 ('experience',
  'Senior Backend Engineer, Northstar Health (2021 - Present)\n- Built FastAPI services for patient data workflows on AWS EC'),
 ('education', 'B.S. Computer Science, Northeastern University, 2017')]

## 4. Embedding Model

The vector index uses the open-source Sentence Transformers model `sentence-transformers/all-MiniLM-L6-v2`, avoiding any OpenAI API key requirement.

In [5]:
resume_rag.EMBEDDING_MODEL, resume_rag.COLLECTION_NAME, resume_rag.DEFAULT_PERSIST_DIR

('sentence-transformers/all-MiniLM-L6-v2',
 'resume_sections',
 'c:\\Airtribe\\BackendJavaTrack_BEL_C19\\llm-file-tools\\chroma_db')

## 5. Chroma Retrieval Example

Semantic retrieval uses the full job description as the query. More chunks are retrieved than final candidates because multiple chunks can belong to the same resume.

In [6]:
jd = open(os.path.join(PROJECT_DIR, 'job_descriptions', 'python_backend.txt'), encoding='utf-8').read()
requirements = extract_job_requirements(jd)
requirements

{'skills': ['Python', 'FastAPI', 'PostgreSQL', 'AWS', 'Docker'],
 'required_skills': ['Python', 'FastAPI', 'PostgreSQL', 'AWS', 'Docker'],
 'min_experience_years': 5}

## 6. Complete Job Matching Example

The final matcher groups chunks by candidate, combines semantic relevance with deterministic skill overlap and experience fit, then validates must-have requirements.

In [7]:
result = match_job_description(jd, top_k_candidates=5, retrieval_k=50)
[(m['candidate_name'], m['match_score'], m['eligible'], m['matched_skills']) for m in result['top_matches']]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[('Isabella Romero',
  85,
  True,
  ['Python', 'FastAPI', 'PostgreSQL', 'AWS', 'Docker']),
 ('Emma Wilson',
  83,
  True,
  ['Python', 'FastAPI', 'PostgreSQL', 'AWS', 'Docker']),
 ('John Doe', 81, True, ['Python', 'FastAPI', 'PostgreSQL', 'AWS', 'Docker']),
 ('Lena Muller',
  78,
  True,
  ['Python', 'FastAPI', 'PostgreSQL', 'AWS', 'Docker']),
 ('Kevin Zhou', 73, False, ['Python', 'FastAPI', 'PostgreSQL', 'Docker'])]

## 7. Evaluation Results

The evaluation script compares retrieved candidate names against the manually curated golden dataset and measures latency with `time.perf_counter()`.

In [8]:
metrics = evaluation.evaluate()
{k: v for k, v in metrics.items() if k != 'per_job_results'}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'job_count': 6,
 'top_1_hit_rate': 1.0,
 'top_3_hit_rate': 1.0,
 'top_5_recall': 1.0,
 'top_10_recall': 1.0,
 'average_latency_seconds': 5.996297449998868}

## 8. Analysis

- Section-aware chunking was chosen because resume sections have meaning: skills, experience, education, and summaries should not be mixed unless a section is genuinely long.
- Semantic retrieval alone is not enough for skills because embeddings can retrieve generally similar profiles while missing exact requirements such as `FastAPI` or `PostgreSQL`.
- Hybrid semantic + keyword matching helps because semantic retrieval finds relevant career context, while exact skill checks validate critical technologies deterministically.
- Latency is mainly driven by embedding the job description and querying Chroma. Local model cache availability matters; offline/cache mode avoids slow network retries.
- Limitations: the parser is rule-based, skills come from a fixed alias list, experience extraction is approximate, and the scoring formula is heuristic rather than learned from hiring outcomes.

## Semantic vs Hybrid Matching

The Chroma query supplies semantic evidence at the chunk level. The final hybrid matcher then adds skill overlap, experience fit, and must-have validation. This can demote semantically plausible candidates who lack required technologies.